# Circuit Synthesis with the Gap Differential Port

This notebook mirrors the *AC Circuit Synthesis* section of [`nbs/palace_inductor.ipynb`](palace_inductor.ipynb) but replaces the two **hand-built interlayer ports** feeding a global guard ring with a single Palace `gap` lumped port — a sheet spanning the slot directly **between the two floating inductor terminals**. The guard ring is kept identical, so this is the *controlled port-only comparison* that [`nbs/palace_inductor_port_comparison.ipynb`](palace_inductor_port_comparison.ipynb) (issue [#257](https://github.com/gdsfactory/gsim/issues/257)) called for.

**Question addressed (issue [#272](https://github.com/gdsfactory/gsim/issues/272)):** with the gap port, the port terminal voltage *is* the differential terminal voltage. Do Palace's synthesized circuit-matrix readouts (`rom-*.csv`) become meaningful RLC parameters directly — instead of the "undefined" series inductance the interlayer two-port produced?

**Requirements:** Notebook dependencies: `uv sync --group palace-inductor` · local run needs Palace >= 0.17 (default runtime v0.18.0).

In [ ]:
import numpy as np
import gdsfactory as gf
from ihp import PDK

PDK.activate()

# Identical geometry to nbs/palace_inductor.ipynb: single-turn spiral + Metal1 guard ring.
c = gf.components.inductor(
    width=2,
    space=2.1,
    diameter=50,
    turns=1,
    layer_metal="TopMetal2drawing",
    layer_inductor="INDdrawing",
    layer_metal_pin="TopMetal2drawing",
    layers_no_fill=("NoMetFillerdrawing",),
).copy()

bbox = c.bbox()
xmin, ymin, xmax, ymax = bbox.left, bbox.bottom, bbox.right, bbox.top
margin_inner, overlap = 15.0, 0.5
ring_sections = (
    (
        (margin_inner + overlap, ymax - ymin),
        (xmin + margin_inner / 2 + overlap / 2, (ymax + ymin) / 2),
    ),
    (
        (margin_inner + overlap, ymax - ymin),
        (xmax - margin_inner / 2 - overlap / 2, (ymax + ymin) / 2),
    ),
    (
        (xmax - xmin, margin_inner + overlap),
        ((xmax + xmin) / 2, ymax - margin_inner / 2 - overlap / 2),
    ),
    (
        (xmax - xmin, margin_inner + overlap),
        ((xmax + xmin) / 2, ymin + margin_inner / 2 + overlap / 2),
    ),
)
for size, center in ring_sections:
    c.add_ref(
        gf.components.rectangle(size=size, layer="Metal1drawing", centered=True)
    ).move(center)

# The gap port: a synthetic electrical port spanning the slot between P1 and P2.
first = np.asarray(c.ports["P1"].center, dtype=float)
second = np.asarray(c.ports["P2"].center, dtype=float)
assert np.isclose(first[1], second[1]), "gap port requires aligned terminals"
gap_width_um = float(
    abs(second[0] - first[0])
    - (float(c.ports["P1"].width) + float(c.ports["P2"].width)) / 2
)
c.add_port(
    name="Pdiff",
    center=tuple((first + second) / 2),
    width=gap_width_um,
    orientation=0,
    layer="TopMetal2drawing",
    port_type="electrical",
)
print(f"gap width between terminals: {gap_width_um:.2f} um")

cc = c.copy()
c.draw_ports()
c.plot()

## Configure and run with circuit synthesis

One adaptive driven sweep produces **both** the S-parameters and the
synthesized L/R/C circuit (`AdaptiveCircuitSynthesis`). Committed outputs from a
previous local Palace v0.18.0 run are reused when present (see
`provenance.json`); delete `nbs/data/inductor/circuit_synthesis_gap/` to
re-run locally.

In [ ]:
import json
import re
import subprocess
from pathlib import Path

from gsim.palace import DrivenSim
from gsim.palace import load_sparams
from gsim.palace.circuit import MIN_PALACE_VERSION
from gsim.palace.runtime import resolve_palace_binary

cache_dir = Path("nbs/data/inductor/circuit_synthesis_gap")
use_cache = (cache_dir / "rom-Linv-re.csv").exists()


def _palace_version(binary):
    if binary is None:
        return "unknown"
    out = subprocess.run(
        [str(binary), "--version"], capture_output=True, text=True, check=False
    ).stdout
    match = re.search(r"v\d+\.\d+\.\d+", out)
    return match.group(0) if match else "unknown"


if use_cache:
    provenance = json.loads((cache_dir / "provenance.json").read_text())
    palace_version = provenance["palace_version"]
    cs_results = load_sparams(cache_dir)
    cs_source = cache_dir
    print(f"Loaded cached gap-port circuit-synthesis results (Palace {palace_version})")
else:
    palace_version = _palace_version(resolve_palace_binary())
    print(f"Running locally with Palace {palace_version} ...")

    sim_gap = DrivenSim()
    sim_gap.set_output_dir("./palace-sim-inductor-gap-cs")
    sim_gap.set_geometry(cc)
    sim_gap.set_stack(substrate_thickness=180.0, include_substrate=True)
    sim_gap.add_port("Pdiff", layer="topmetal2", impedance=50.0, geometry="gap")
    sim_gap.set_driven(fmin=10e9, fmax=200e9, num_points=201, circuit_synthesis=True)
    sim_gap.set_airbox(margin_x=50, margin_y=50, z_above=50, z_below=5)
    print("validate:", sim_gap.validate_config())
    sim_gap.mesh(preset="default", refined_mesh_size=1.5)
    sim_gap.write_config()
    cs_results = sim_gap.run_local(verbose=False)
    cs_source = cs_results
    print("Done.")

parsed = tuple(int(x) for x in re.findall(r"\d+", palace_version)[:3])
if len(parsed) == 3 and parsed < tuple(int(x) for x in MIN_PALACE_VERSION.split(".")):
    print(
        f"WARNING: Palace {palace_version} < {MIN_PALACE_VERSION} lacks circuit synthesis"
    )
else:
    print("Palace runtime version:", palace_version)

cs_f = cs_results.freq * 1e9  # GHz -> Hz

In [ ]:
import numpy as np

from gsim.palace import load_circuit_synthesis

circuit = load_circuit_synthesis(cs_source, port_map={1: "Pdiff"})
print(circuit)
print("port nodes:    ", circuit.port_labels)
print(
    "interior nodes:", len(circuit.internal_indices), "(adaptive-sweep sample states)"
)

k = circuit.port_indices[0]
l_raw = 1.0 / circuit.L_inv[k, k].real
c_raw = circuit.C[k, k].real
print()
print(f"raw port-block readout (what Palace's matrices give for the port row):")
print(
    f"  L_inv[port,port] = {circuit.L_inv[k, k]:.3e} 1/H  ->  L_raw = {l_raw * 1e12:.1f} pH"
)
print(f"  C[port,port]     = {c_raw * 1e15:.3f} fF")
print(
    f"  port termination = 1/R_inv = {1.0 / circuit.port_loads['port_1_re']['R_inv'].diagonal()[k].real:.1f} Ohm"
)

## Method A &mdash; raw port readout: small, physical &mdash; and still not the device value

For the two **interlayer** ports of the main notebook the raw port block
contained **no device parameters**: `L⁻¹` port↔port coupling is exactly 0, so
the series inductance readout is "undefined"; the port diagonals carry only the
*port-region self-terms* (0.67/0.62 pH for the 2 &mu;m via pads &mdash; plausible for
such small surfaces, but ~200&times; below the device) plus aF-scale pad caps.

With the **gap** port the same is true and the unit scale confirms it: the raw
port self-term is **~0.061 pH** for the 2.1 &mu;m slot (also physically small,
~2000&times; below the device), with a 0.019 fF pad and the 50 &Omega; termination.
Neither port definition hands you the inductance we are after: the device value
is *emergent* &mdash; it exists only in the network after the interior nodes are
eliminated (Method B).

## Method B &mdash; condensed one-port device model & (no Z_diff algebra needed)

Eliminating the interior nodes (Schur complement) and subtracting the shipped
50 Ω termination gives the bare-device admittance directly &mdash; for a one-port
setup, $Z_{11}$ *is* the differential impedance, so `Z11 - Z12 - Z21 + Z22`
bookkeeping disappears (`differential_impedance` falls back to the
driving-point impedance for `N == 1`). We compare against the FEM `S11`
converted with the same 50 Ω reference.

In [ ]:
import matplotlib.pyplot as plt

from gsim.palace import differential_impedance, fit_rlc

z_circ = differential_impedance(circuit.port_impedance(cs_f))

# FEM S11 -> Z11 (same 50 Ohm reference), no skrf needed.
import pandas as pd

if use_cache:
    df = pd.read_csv(cs_source / "port-S.csv")
else:
    df = pd.read_csv(cs_results.files["port-S.csv"])
df.columns = df.columns.str.strip()
s11 = 10 ** (df["|S[1][1]| (dB)"].to_numpy() / 20) * np.exp(
    1j * np.deg2rad(df["arg(S[1][1]) (deg.)"].to_numpy())
)
z_fem = 50.0 * (1 + s11) / (1 - s11)

w = 2 * np.pi * cs_f
low = cs_f < 30e9
l_circ = float(np.mean(np.imag(z_circ[low]) / w[low]))
l_fem = float(np.mean(np.imag(z_fem[low]) / w[low]))
s_err = float(np.abs(circuit.s_parameters(cs_f)[:, 0, 0] - s11).max())
print(f"L (low-freq)  circuit: {l_circ * 1e12:.3f} pH | FEM S: {l_fem * 1e12:.3f} pH")
print(
    f"R (low-freq)  circuit: {np.mean(np.real(z_circ[low])):.4f} Ohm | FEM S: {np.mean(np.real(z_fem[low])):.4f} Ohm"
)
print(f"max |S_circuit - S_fem| = {s_err:.3e}")

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
axes[0].plot(cs_f / 1e9, np.abs(z_fem), ".", ms=3, label="FEM S11 -> Z")
axes[0].plot(cs_f / 1e9, np.abs(z_circ), label="Exported circuit (condensed)")
axes[0].set_yscale("log")
axes[0].set_ylabel("|Z| [Ohm]")
axes[0].set_xlabel("f [GHz]")
axes[0].legend()
axes[1].plot(cs_f / 1e9, np.angle(z_fem, deg=True), ".", ms=3, label="FEM S11 -> Z")
axes[1].plot(
    cs_f / 1e9, np.angle(z_circ, deg=True), label="Exported circuit (condensed)"
)
axes[1].set_ylabel("arg(Z) [deg]")
axes[1].set_xlabel("f [GHz]")
axes[1].legend()
plt.tight_layout()
plt.show()

## Method C &mdash; gap port vs the two-port setup (identical surrounding metal)

Both setups keep the same device and guard ring, so differences are pure
port-definition effects. Comparing the exported circuits (condensed, bare)
either side of the same frequency grid, plus the same one-pole fit:

In [ ]:
two_source = Path("nbs/data/inductor/circuit_synthesis")
circuit_2p = load_circuit_synthesis(two_source, port_map={1: "P1", 2: "P2"})
z_2p = differential_impedance(circuit_2p.port_impedance(cs_f))

import pandas as pd

fit_circ = fit_rlc(cs_f, z_circ, solver="scipy")
fit_fem = fit_rlc(cs_f, z_fem, solver="scipy")
fit_2p = circuit_2p.fit_rlc(cs_f, solver="scipy")
l_2p = float(np.mean(np.imag(z_2p[low]) / w[low]))

print(
    f"{'extraction':26s} {'R [Ohm]':>9s} {'L [pH]':>9s} {'C [fF]':>8s} "
    f"{'f0 [GHz]':>9s} {'Q':>7s}"
)
for name, ft, lf in (
    ("gap circuit", fit_circ, l_circ),
    ("gap FEM S11", fit_fem, l_fem),
    ("two-port circuit", fit_2p, l_2p),
):
    print(
        f"{name:26s} {ft.R:9.4f} {ft.L * 1e12:9.3f} {ft.C * 1e15:8.3f} "
        f"{ft.f0 / 1e9:9.3f} {ft.Q:7.2f}   (L_lf = {lf * 1e12:.1f} pH)"
    )

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
axes[0].plot(cs_f / 1e9, np.abs(z_2p), label="two-port interlayer (Z_diff)")
axes[0].plot(cs_f / 1e9, np.abs(z_fem), ".", ms=3, label="gap port (FEM S11 -> Z)")
axes[0].set_yscale("log")
axes[0].set_ylabel("|Z| [Ohm]")
axes[0].set_xlabel("f [GHz]")
axes[0].legend()
axes[1].plot(cs_f / 1e9, np.angle(z_2p, deg=True), label="two-port interlayer")
axes[1].plot(cs_f / 1e9, np.angle(z_fem, deg=True), ".", ms=3, label="gap port")
axes[1].set_ylabel("arg(Z) [deg]")
axes[1].set_xlabel("f [GHz]")
axes[1].legend()
plt.tight_layout()
plt.show()

## Validation

Acceptance criteria (issue [#272](https://github.com/gdsfactory/gsim/issues/272)):
the exported-circuit response must match the FEM response within **0.01**
absolute complex S, and the compact parameters extracted from the two port
definitions must be consistent (the port definition itself shifts the
low-frequency reference by ~6 %, reproducing the effect seen in the
port-comparison notebook).

In [ ]:
assert s_err < 1e-2, f"exported gap circuit deviates from FEM: max |dS| = {s_err:.3e}"
assert abs(l_circ - l_fem) / l_fem < 1e-3, "low-frequency L mismatch (circuit vs FEM)"
assert abs(fit_circ.R - fit_fem.R) < 0.02 * fit_fem.R
assert abs(fit_circ.L - fit_fem.L) / fit_fem.L < 0.01
assert abs(fit_circ.f0 - fit_fem.f0) / fit_fem.f0 < 0.005
# Port-definition consistency (gap vs two-port, same device and ring):
assert abs(fit_circ.L - fit_2p.L) / fit_2p.L < 0.02
assert abs(fit_circ.f0 - fit_2p.f0) / fit_2p.f0 < 0.01
assert abs(l_circ - l_2p) / l_2p < 0.10  # port-definition shift, see text

print("All gap-port circuit-synthesis validation checks passed.")
print(f"  max |S_circuit - S_fem| = {s_err:.3e}  (< 0.01)")
print(
    f"  raw port self-term  L = {l_raw * 1e12:.3f} pH  (Two-port setup: none/'undefined')"
)
print(
    f"  condensed device    L = {l_circ * 1e12:.1f} pH, "
    f"fit: R={fit_circ.R:.3f} Ohm, L={fit_circ.L * 1e12:.2f} pH, "
    f"C={fit_circ.C * 1e15:.2f} fF, f0={fit_circ.f0 / 1e9:.2f} GHz, Q={fit_circ.Q:.1f}"
)
print(f"  (Palace {palace_version}; rom-*.csv output requires >= {MIN_PALACE_VERSION})")

## Interpretation

- The **gap port makes the port terminal differential by construction**: $S_{11}
  \to Z_{11}$ is directly the terminal impedance; the `Z11 - Z12 - Z21 + Z22`
  step (and its sensitivity) disappears.
- The "Palace returns meaningful RLC directly" hypothesis is **refuted for both
  port definitions**: the raw port self-terms are small port-local parasitics
  (0.62&ndash;0.67 pH per 2 &mu;m interlayer pad; 0.061 pH for the 2.1 &mu;m slot),
  never the device's 125&ndash;132 pH differential inductance. Palace exports the
  machine (projected-field coordinates), not compact values.
- Condensation on the exported circuit is the supported extraction: max
  |&#916;S| = 5.8e-4 vs FEM, L to 0.02 %, and the one-pole fit reproduces the FEM
  fit to &lt;0.1 % in every parameter.
- Changing the port definition *does* move the numbers: the low-frequency
  inductance differs by ~6 % between the gap and two-terminal setups (125 vs
  132 pH) &mdash; the voltage/current references differ, reproducing the
  resonance shifts of the port-comparison notebook &mdash; while the resonance-region
  one-pole parameters stay within ~1 %.